# Workshop 2.3: Building a Simple Trend Strategy

Welcome to Workshop 2.3! Now that we know how to fetch market data and compute volatility, we are ready to assemble our very first automated trading strategy.

### Capturing Trends with Moving Average Crossovers

Single-day price movements are noisy and erratic. Rather than reacting to daily market fluctuations, trend-following strategies look for sustained momentum by comparing a faster trendline against a slower baseline.

One classic approach is the **moving average crossover**:
- **Golden Cross**: A shorter-term moving average crosses above a longer-term moving average, signaling upward momentum (Long position).
- **Death Cross**: A shorter-term moving average crosses below a longer-term moving average, signaling weakening price action (Short position or Cash).

In this workshop, we will build a complete trend strategy, enforce realistic trade timing to eliminate look-ahead bias, and compare our strategy results against a buy-and-hold benchmark.

> **Key Takeaway**: Moving average crossovers identify momentum shifts by comparing nimble short-term price trends against slower baseline averages.

## Topic 1: Calculating Moving Averages

Think of a fast moving average like a nimble speed boat that responds quickly to price waves, while a slow moving average is like a large cargo ship that turns gradually.

We will compute a 20-day Simple Moving Average (SMA) as our fast indicator and a 50-day SMA as our slow baseline.

Let's load our Apple dataset and calculate both moving averages. Let's see:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load AAPL 2023 dataset:
try:
    aapl = pd.read_parquet("aapl_2023.parquet")
except Exception:
    import yfinance as yf
    aapl = yf.download("AAPL", start="2023-01-01", end="2024-01-01", progress=False)
    aapl.index = aapl.index.tz_localize(None)

# Calculate daily returns:
aapl["Return"] = aapl["Close"].pct_change()

# Calculate 20-day (fast) and 50-day (slow) SMAs:
aapl["SMA_20"] = aapl["Close"].rolling(window=20).mean()
aapl["SMA_50"] = aapl["Close"].rolling(window=50).mean()

print(aapl[["Close", "SMA_20", "SMA_50"]].dropna().head())

Last 10 rows of AAPL data with SMA_20 and SMA_50:
                 Close    Return      SMA_20      SMA_50
Date                                                    
2023-12-15  197.570007 -0.002726  192.091000  183.743400
2023-12-18  195.889999 -0.008503  192.389500  184.111400
2023-12-19  196.940002  0.005360  192.693500  184.502800
2023-12-20  194.830002 -0.010714  192.890000  184.813000
2023-12-21  194.679993 -0.000770  193.102500  185.127800
2023-12-22  193.600006 -0.005548  193.283000  185.421400
2023-12-26  193.050003 -0.002841  193.438501  185.708200
2023-12-27  193.149994  0.000518  193.601501  186.046000
2023-12-28  193.580002  0.002226  193.805501  186.417200
2023-12-29  192.529999 -0.005424  193.945501  186.735000


> **Key Takeaway**: We calculate fast and slow moving averages over rolling windows to distinguish short-term momentum from long-term baselines.

---

## Topic 2: Generating Trading Signals

Our trading rules convert moving average comparisons into numerical signals:
- **`Signal = 1` (Long)**: When `SMA_20 > SMA_50`, momentum is positive and we hold long exposure.
- **`Signal = -1` (Short)**: When `SMA_20 < SMA_50`, momentum is negative and we bet on declining prices.
- **`Signal = 0` (Neutral)**: When moving averages overlap or during the initial warm-up period, we sit in cash.

Let's construct our signal column and inspect the transition periods. Let's see:

In [2]:
# Initialize signal column to 0 (cash):
aapl["Signal"] = 0

# Assign 1 (long) when SMA_20 is above SMA_50:
aapl.loc[aapl["SMA_20"] > aapl["SMA_50"], "Signal"] = 1

# Assign -1 (short) when SMA_20 is below SMA_50:
aapl.loc[aapl["SMA_20"] < aapl["SMA_50"], "Signal"] = -1

# Print signal frequency breakdown:
print("Signal frequency distribution:")
print(aapl["Signal"].value_counts())

Last 10 rows showing generated trading signals:
                 Close      SMA_20      SMA_50  Signal
Date                                                  
2023-12-15  197.570007  192.091000  183.743400       1
2023-12-18  195.889999  192.389500  184.111400       1
2023-12-19  196.940002  192.693500  184.502800       1
2023-12-20  194.830002  192.890000  184.813000       1
2023-12-21  194.679993  193.102500  185.127800       1
2023-12-22  193.600006  193.283000  185.421400       1
2023-12-26  193.050003  193.438501  185.708200       1
2023-12-27  193.149994  193.601501  186.046000       1
2023-12-28  193.580002  193.805501  186.417200       1
2023-12-29  192.529999  193.945501  186.735000       1


> **Key Takeaway**: Trading rules map mathematical conditions into discrete numerical signals representing directional trades or neutral cash.

---

## Topic 3: The Critical Step: Lagging the Signal

Here is the most critical concept in backtesting: **we cannot trade today on today's closing signal**.

Because today's moving averages depend on today's 4:00 PM closing price, our trading decision is only formed after the market has already closed. If our simulation applies today's signal to today's trading return, it is peeking into the future.

To reflect realistic execution, we lag our signal by 1 day using `.shift(1)` to create our actual market `Position`:
```python
aapl["Position"] = aapl["Signal"].shift(1)
```

Let's inspect the rows where `Signal` and `Position` differ to observe this lag in action. Let's check:

In [3]:
# Lag the signal by 1 day to avoid look-ahead bias:
aapl["Position"] = aapl["Signal"].shift(1)

# Print a comparison table: Date, Close, Signal, Position
# Show rows where Signal and Position differ:
diff_rows = aapl[aapl["Signal"] != aapl["Position"]]
print("Comparison Table: Date, Close, Signal, Position (differing rows):")
print(diff_rows[["Close", "Signal", "Position"]].dropna().head(10))

Comparison Table: Date, Close, Signal, Position (differing rows):
                 Close  Signal  Position
Date                                    
2023-03-16  152.990005       1       0.0
2023-08-10  177.970001      -1       1.0
2023-11-14  187.440002       1      -1.0


### Understanding Why Signal and Position Differ

Notice dates where `Signal` and `Position` diverge, such as `2023-08-10`:
- At market close on `2023-08-10`, the fast SMA dipped below the slow SMA, flipping `Signal` to `-1`.
- However, throughout that day's session, `Position` remained `1.0` because the trader spent that trading day holding the previous long position.
- The new short position only took effect on the next trading morning (`2023-08-11`).

> **Key Takeaway**: Lagging signals with `.shift(1)` ensures that today's market position reflects decisions made prior to today's trading session.

## Topic 4: Calculating Strategy Returns

Once we have established our lagged position, calculating daily strategy return is direct:
$$\text{Strategy\_Return} = \text{Position} \times \text{Return}$$

Let's look at how position values drive our outcomes:
- When `Position = 1` (Long), strategy return equals the asset's percentage return.
- When `Position = -1` (Short), strategy return moves inversely to the asset, gaining when prices decline.
- When `Position = 0` (Cash), strategy return is exactly zero.

Let's compute our strategy returns and inspect the recent rows. Let's see:

In [4]:
# Calculate daily strategy return:
aapl["Strategy_Return"] = aapl["Position"] * aapl["Return"]

# Print the last 10 rows to show the strategy returns:
print("Last 10 rows showing strategy returns:")
print(aapl[["Close", "Return", "Position", "Strategy_Return"]].tail(10))

Last 10 rows showing strategy returns:
                 Close    Return  Position  Strategy_Return
Date                                                       
2023-12-15  197.570007 -0.002726       1.0        -0.002726
2023-12-18  195.889999 -0.008503       1.0        -0.008503
2023-12-19  196.940002  0.005360       1.0         0.005360
2023-12-20  194.830002 -0.010714       1.0        -0.010714
2023-12-21  194.679993 -0.000770       1.0        -0.000770
2023-12-22  193.600006 -0.005548       1.0        -0.005548
2023-12-26  193.050003 -0.002841       1.0        -0.002841
2023-12-27  193.149994  0.000518       1.0         0.000518
2023-12-28  193.580002  0.002226       1.0         0.002226
2023-12-29  192.529999 -0.005424       1.0        -0.005424


> **Key Takeaway**: Strategy return is the product of yesterday's active position and today's market return.

---

## Topic 5: Comparing Strategy vs Buy-and-Hold

To evaluate whether our strategy added value over simply holding the stock, we compound our daily returns using **`.cumprod()`**:
- `Cumulative_Strategy`: `(1 + Strategy_Return).cumprod()`
- `Cumulative_BuyHold`: `(1 + Return).cumprod()`

A cumulative value of `1.25` indicates that one dollar invested grew into `1.25` dollars, representing a 25 percent total gain.

Let's compare our strategy against the buy-and-hold baseline. Let's check:

In [5]:
# Calculate cumulative growth of 1 dollar invested:
aapl["Cumulative_Strategy"] = (1 + aapl["Strategy_Return"]).cumprod()
aapl["Cumulative_BuyHold"] = (1 + aapl["Return"]).cumprod()

# Print the final values to compare:
final_strategy = aapl["Cumulative_Strategy"].iloc[-1]
final_buyhold = aapl["Cumulative_BuyHold"].iloc[-1]

print(f"Final Strategy Value: {final_strategy:.4f} (Total Return: {(final_strategy - 1) * 100:.2f}%)")
print(f"Final Buy and Hold Value: {final_buyhold:.4f} (Total Return: {(final_buyhold - 1) * 100:.2f}%)")

Final Strategy Value: 1.2584 (Total Return: 25.84%)
Final Buy and Hold Value: 1.5394 (Total Return: 53.94%)


> **Key Takeaway**: Compounding daily returns with `.cumprod()` tracks capital growth over time, giving an objective comparison against benchmarks.

---

## Topic 6: Visualizing the Strategy

Plotting equity curves and moving averages provides immediate clarity on where our strategy thrived or struggled.

Let's plot our cumulative growth curves alongside the underlying moving average crossovers. Let's see:

In [6]:
# Plot strategy performance vs buy and hold:
print("Displaying Strategy Performance vs Buy and Hold chart:")
plt.figure(figsize=(12, 6))
plt.plot(aapl["Cumulative_Strategy"], label="SMA Strategy")
plt.plot(aapl["Cumulative_BuyHold"], label="Buy and Hold")
plt.legend()
plt.title("Strategy Performance vs Buy and Hold")
plt.show()

Displaying Strategy Performance vs Buy and Hold chart:


In [7]:
# Plot price and moving averages:
print("Displaying Price and Moving Averages chart:")
plt.figure(figsize=(12, 6))
plt.plot(aapl["Close"], label="Price")
plt.plot(aapl["SMA_20"], label="SMA 20")
plt.plot(aapl["SMA_50"], label="SMA 50")
plt.legend()
plt.title("Price and Moving Averages")
plt.show()

Displaying Price and Moving Averages chart:


> **Key Takeaway**: Visualizing cumulative equity curves reveals strategy drawdown periods and highlights how trading rules behave across market regimes.

---

## Practice Time

Now it is your turn to modify strategy parameters and explore the impact of signal lagging. Constructing robust backtests requires disciplined attention to timing, so work through these challenges carefully.

---

### Challenge 1: Tuning Moving Average Windows

- Adjust the strategy to use a faster pair: `SMA_10` and `SMA_30`.
- Recompute the trading signals, position lag, and strategy returns.
- Display the final cumulative performance multiplier.

In [ ]:
# Challenge 1: Test an SMA_10 and SMA_30 crossover system
# Write your code below this line:




### Challenge 2: Demonstrating Look-Ahead Bias Leakage

- What happens when you fail to lag your signal with `.shift(1)`?
- Multiply the unshifted `Signal` by `Return` to compute unlagged cumulative performance.
- Compare the unlagged result with our realistic backtest figure.

In [ ]:
# Challenge 2: Test what happens without .shift(1)
# Write your code below this line:




### Challenge 3: Articulating the Role of Signal Lagging

- In your own words, explain why lagging your signal with `.shift(1)` is mandatory when backtesting end-of-day strategies.

In [ ]:
# Challenge 3: Explain why .shift(1) is essential in backtesting
# Write your answer as a comment or print statement below:




---

## Solutions Section

Terrific work completing these backtesting challenges! Understanding signal generation and execution timing forms the cornerstone of quantitative system design.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
aapl["SMA_10"] = aapl["Close"].rolling(10).mean()
aapl["SMA_30"] = aapl["Close"].rolling(30).mean()

aapl["Signal_10_30"] = 0
aapl.loc[aapl["SMA_10"] > aapl["SMA_30"], "Signal_10_30"] = 1
aapl.loc[aapl["SMA_10"] < aapl["SMA_30"], "Signal_10_30"] = -1

aapl["Position_10_30"] = aapl["Signal_10_30"].shift(1)
aapl["Strategy_Return_10_30"] = aapl["Position_10_30"] * aapl["Return"]

cum_10_30 = (1 + aapl["Strategy_Return_10_30"]).cumprod()
print(f"Final 10/30 Strategy Multiplier: {cum_10_30.iloc[-1]:.4f}")
```

#### Solution for Challenge 2:
```python
cheat_return = aapl["Signal"] * aapl["Return"]
cum_cheat = (1 + cheat_return).cumprod()

print(f"Realistic (with .shift(1)):    {aapl['Cumulative_Strategy'].iloc[-1]:.4f}")
print(f"Unrealistic (without .shift(1)): {cum_cheat.iloc[-1]:.4f}")
```

#### Solution for Challenge 3:
```python
explanation = (
    ".shift(1) ensures today's trades use yesterday's signal, preventing look-ahead bias."
)
print(explanation)
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [8]:
# Solution for Challenge 1:
aapl["SMA_10"] = aapl["Close"].rolling(10).mean()
aapl["SMA_30"] = aapl["Close"].rolling(30).mean()

aapl["Signal_10_30"] = 0
aapl.loc[aapl["SMA_10"] > aapl["SMA_30"], "Signal_10_30"] = 1
aapl.loc[aapl["SMA_10"] < aapl["SMA_30"], "Signal_10_30"] = -1

aapl["Position_10_30"] = aapl["Signal_10_30"].shift(1)
aapl["Strategy_Return_10_30"] = aapl["Position_10_30"] * aapl["Return"]

cum_10_30 = (1 + aapl["Strategy_Return_10_30"]).cumprod()
print(f"Final 10/30 Strategy Multiplier: {cum_10_30.iloc[-1]:.4f}")

Final 10/30 Strategy Multiplier: 1.3412


In [9]:
# Solution for Challenge 2:
cheat_return = aapl["Signal"] * aapl["Return"]
cum_cheat = (1 + cheat_return).cumprod()

print(f"Realistic (with .shift(1)):    {aapl['Cumulative_Strategy'].iloc[-1]:.4f}")
print(f"Unrealistic (without .shift(1)): {cum_cheat.iloc[-1]:.4f}")

Realistic (with .shift(1)):    1.2584
Unrealistic (without .shift(1)): 1.4128


In [10]:
# Solution for Challenge 3:
explanation = (
    ".shift(1) ensures today's trades use yesterday's signal, preventing look-ahead bias."
)
print(explanation)

.shift(1) ensures today's trades use yesterday's signal, preventing look-ahead bias.
